In [9]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

mat_path = Path('/content/drive/MyDrive/EECE_798K_Project/data/p5848_datatable.mat')

In [10]:
one_d = {}

with h5py.File(mat_path, 'r') as f:
    refs = f['#refs#']

    for k in refs.keys():
        obj = refs[k]

        if hasattr(obj, 'dtype') and hasattr(obj, 'shape'):
            if str(obj.dtype) == 'float64' and len(obj.shape) == 2 and obj.shape[0] == 1:
                if obj.shape[1] > 1000:
                    one_d[k] = obj[()].flatten()

print("Found 1D signals:", len(one_d))
print(list(one_d.keys()))

Found 1D signals: 35
['A', 'B', 'H', 'I', 'J', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [11]:
preview = {}

for k, arr in one_d.items():
    preview[k] = {
        "min": float(np.nanmin(arr)),
        "max": float(np.nanmax(arr)),
        "first5": arr[:5]
    }

for k, v in preview.items():
    print(f"\n{k}")
    print("min:", v["min"])
    print("max:", v["max"])
    print("first5:", v["first5"])


A
min: -974.9701341862403
max: 831.4523860435235
first5: [0. 0. 0. 0. 0.]

B
min: -361.3628214683213
max: 321.7956107165714
first5: [0. 0. 0. 0. 0.]

H
min: 3066.1068490769094
max: 3278.6058791738787
first5: [nan nan nan nan nan]

I
min: 533.5642706405284
max: 564.5386780730674
first5: [nan nan nan nan nan]

J
min: -0.0018031555252487606
max: 0.0530578168201837
first5: [nan nan nan nan nan]

L
min: -109.20462271018839
max: 122.47757060766243
first5: [0. 0. 0. 0. 0.]

M
min: -79.87462427740337
max: 77.88174443018607
first5: [0. 0. 0. 0. 0.]

N
min: -0.9943245546301068
max: 1.1659558150593152
first5: [-0. -0. -0. -0. -0.]

O
min: -3.1100116017760715e-09
max: 1.4507661877757209e-08
first5: [2.09120165e-09 2.08886738e-09 2.08548102e-09 2.09280746e-09
 2.07959327e-09]

P
min: 814657.851791053
max: 1175479.1098926857
first5: [ 995893.63460919  989026.91490871 1025716.89410567  993116.81967756
 1004691.44742953]

Q
min: -2.054886956591156e-16
max: 9.610585586952666e-16
first5: [1.35609203e-1

In [16]:
mapping = {
    'time': 'z',
    'tau': 'B',
    'v_int': 'f',
    'sigmaN': 'h',
    'mu': 'k'
}

In [17]:
with h5py.File(mat_path, 'r') as f:
    refs = f['#refs#']

    df_core = pd.DataFrame({
        name: refs[key][()].flatten()
        for name, key in mapping.items()
    })

print(df_core.shape)
print(df_core.head())

(3708340, 5)
       time  tau         v_int     sigmaN        mu
0  8821.381  0.0  3.818795e-09  19.018319 -0.604563
1  8821.382  0.0  3.796373e-09  19.007830 -0.604519
2  8821.383  0.0  3.788739e-09  18.987750 -0.604472
3  8821.384  0.0  3.806035e-09  18.972081 -0.604540
4  8821.385  0.0  3.808619e-09  18.981770 -0.604562


In [18]:
print(df_core.describe().T)

            count          mean           std           min           25%  \
time    3708340.0  1.067555e+04  1.070506e+03  8.821381e+03  9.748466e+03   
tau     3708340.0  5.122013e+00  1.287697e+01 -3.613628e+02  0.000000e+00   
v_int   3708340.0  2.088889e-09  3.861305e-09 -9.122218e-08  1.541667e-09   
sigmaN  3708340.0  1.899749e+01  1.431175e-02  1.887923e+01  1.898773e+01   
mu      3708340.0 -3.318361e-01  1.528762e-01 -6.048846e-01 -4.483178e-01   

                 50%           75%           max  
time    1.067555e+04  1.160264e+04  1.252972e+04  
tau     3.060484e+00  8.716736e+00  3.217956e+02  
v_int   1.885286e-09  2.554707e-09  9.425053e-08  
sigmaN  1.899728e+01  1.900704e+01  1.916423e+01  
mu     -2.483574e-01 -2.328532e-01 -2.009516e-01  


In [19]:
# remove flat initial region
df_core = df_core[df_core['tau'] != 0].reset_index(drop=True)

print(df_core.shape)

(3222677, 5)


In [20]:
#derivatives

dt = np.mean(np.diff(df_core['time']))

df_core['tau_dot'] = np.gradient(df_core['tau'], dt)
df_core['v_dot'] = np.gradient(df_core['v_int'], dt)

In [21]:
csv_path = '/content/drive/MyDrive/EECE_798K_Project/data/p5848_clean.csv'

df_core.to_csv(csv_path, index=False)

print("Saved:", csv_path)

Saved: /content/drive/MyDrive/EECE_798K_Project/data/p5848_clean.csv


In [43]:
!git add Notebooks/EDA.ipynb

In [44]:
!git status

On branch malek-utah-forge
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   Notebooks/EDA.ipynb



In [45]:
!git commit -m "update EDA notebook"

[malek-utah-forge b6c6353] update EDA notebook
 1 file changed, 1 insertion(+), 1 deletion(-)


In [46]:
!git status

On branch malek-utah-forge
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   Notebooks/EDA.ipynb

no changes added to commit (use "git add" and/or "git commit -a")
